### Training MMIDAS - a coupled mixture VAE model
This notebook guides you through the process of training a mixture variational autoencoder.

In [111]:
import os

if os.getcwd().split(os.sep)[-1] != 'mmidas':
    os.chdir('..')

assert os.getcwd().split(os.sep)[-1] == 'mmidas', "Please run this script from the mmidas directory."

In [112]:
%load_ext autoreload
%autoreload 2
from mmidas.cpl_mixvae import cpl_mixVAE
from mmidas.utils.tools import get_paths
from mmidas.utils.dataloader import load_data, get_loaders, show_summary
from mmidas._utils import set_seeds

import warnings
warnings.filterwarnings("ignore")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


Specify the training parameters.

In [113]:
spec = {
    "n_run": 1,
    "augmentation": False,
    "n_categories": 120,
    "input_dim": 5032,
    "state_dim": 2,
    "n_arm": 2,
    "latent_dim": 10,
    "batch_size": 5000,
    "n_epoch": 3,
    "n_epoch_p": 5,
    "min_con": 0.9,
    "max_prun_it": 2,
    "batch_size:": 5000,
    "lr": 1e-3,
    "seed": 546,
    "device": "mps"
}

set_seeds(spec['seed'])

Load the prepared data (as described in ```1_data_prep.ipynb```) and create training and validation sets.

In [114]:
data_file = "Mouse_ALM-VISp_cpm.h5ad"
dataset = load_data(".." + "/" + "data" + "/" + data_file)
xs_T = dataset['log1p']
train_loader, test_loader, _, = get_loaders(xs_T, batch_size=spec['batch_size'])

print(show_summary(dataset))

Summary
n_cell_types: 115
n_cells: 22365
n_genes: 5032


Create a designated folder to store training files

In [115]:
def show_spec(spec):
    n_run        = spec['n_run']
    n_categories = spec['n_categories']
    state_dim    = spec['state_dim']
    augmentation = spec['augmentation']
    lr           = spec['lr']
    n_arm        = spec['n_arm']
    batch_size   = spec['batch_size']
    n_epoch      = spec['n_epoch']
    n_epoch_p    = spec['n_epoch_p']
    return "_".join(["run"     + "_" + str(n_run),
                     "K"       + "_" + str(n_categories),
                     "Sdim"    + "_" + str(state_dim),
                     "aug"     + "_" + str(augmentation),
                     "lr"      + "_" + str(lr),
                     "n_arm"   + "_" + str(n_arm),
                     "nbatch"  + "_" + str(batch_size),
                     "nepoch"  + "_" + str(n_epoch),
                     "nepochP" + "_" + str(n_epoch_p)])

saving_folder = "results" + "/" + show_spec(spec)
os.makedirs(saving_folder, exist_ok=True)
os.makedirs(saving_folder + "/" + "model", exist_ok=True)

Construct a cpl-mixVAE object and launch its training on the prepared data.

In [ ]:
import torch as th

set_seeds(spec['seed'])

cplMixVAE = cpl_mixVAE(saving_folder=saving_folder, device=spec['device'])
cplMixVAE.init_model(
    n_categories=spec['n_categories'],
    state_dim=spec['state_dim'],
    input_dim=spec['input_dim'],
    lowD_dim=spec['latent_dim'],
    lr=spec['lr'],
    n_arm=spec['n_arm']
)
cplMixVAE.model = cplMixVAE.model.to(spec['device'])

model_file = cplMixVAE.train(
    train_loader=train_loader,
    test_loader=test_loader,
    n_epoch=spec['n_epoch'],
    n_epoch_p=spec['n_epoch_p'],
    min_con=spec['min_con'],
    max_prun_it=spec['max_prun_it']
)

using device: mps
1 epoch = 4 batches
step: 1 | loss: 235351824.00 | loss-rec-0: 44351.00 | loss-rec-1: 44345.59 | loss-joint: 235263120.00 | throughout: 8411.12it/s | dt: 594.45ms
step: 2 | loss: 177792576.00 | loss-rec-0: 44313.18 | loss-rec-1: 44298.13 | loss-joint: 177703968.00 | throughout: 58429.02it/s | dt: 85.57ms
step: 3 | loss: 145003344.00 | loss-rec-0: 44248.68 | loss-rec-1: 44222.77 | loss-joint: 144914880.00 | throughout: 55639.62it/s | dt: 89.86ms
step: 4 | loss: 109352408.00 | loss-rec-0: 44207.46 | loss-rec-1: 44167.94 | loss-joint: 109264032.00 | throughout: 62764.36it/s | dt: 79.66ms
step: 5 | loss: 88298928.00 | loss-rec-0: 44198.97 | loss-rec-1: 44143.57 | loss-joint: 88210584.00 | throughout: 62506.95it/s | dt: 79.99ms
step: 6 | loss: 67111600.00 | loss-rec-0: 44102.91 | loss-rec-1: 44030.35 | loss-joint: 67023464.00 | throughout: 48666.97it/s | dt: 102.74ms
step: 7 | loss: 54942160.00 | loss-rec-0: 43982.30 | loss-rec-1: 43889.97 | loss-joint: 54854288.00 | throu

In [ ]:
from mmidas.train import train_mmidas
from mmidas.model import make_mmidas

# TODO: enhance: more descriptive variable names
es = { # Experiment specification
    'is_augment': False,
    'input_dim': 5032,
    'fc_dim': 100,
    'lowD_dim': 10,
    'state_dim': 2,
    'n_categories': 120,
    'n_arms': 2,
    'n_pr': 0,
    'n_epochs': 100000,
    # 'n_epochs': 3,
    'temp': 1.0,
    'eps': 1e-8,
    'is_ref_prior': False,
    'x_drop': 0.5,
    's_drop': 0.2,
    'lam': 1,
    'lam_pc': 1,
    'tau': 0.005,
    'beta': 1.0,
    'is_hard': False,
    'is_variational': True,
    'momentum': 0.01,
    'loss': 'MSE',
    'lr': 1e-3,
    'c_prior': 0,
    'c_onehot': 0,
    'min_con': 0.5,
    'max_prun_it': 0,
    'device': 'mps',
    'seed': 546
}

set_seeds(es['seed'])

model = make_mmidas(es)
solver = th.optim.Adam(model.parameters(), lr=es['lr'])

train_mmidas(model, solver, train_loader, test_loader, es)

Working directly with command line, you have the option to train the model using a Python file, such as ```tutorial/train_unimodal.py``` as follows.

```
python train_unimodal.py --n_epoch 10 --n_epoch_p 5 --max_prun_it 2
```
or
```
python train_unimodal.py --n_epoch 10 --n_epoch_p 5 --max_prun_it 2 --device 'cuda'
```

In [ ]:
from mmidas._utils import params_is_equal

params_is_equal(cplMixVAE.model.state_dict(), model.state_dict())
